In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install --upgrade --force-reinstall transformers qwen-vl-utils

Unconstrained Prompt

In [ ]:
import os
import json
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import time

folder_path = '/content/drive/MyDrive/allRotatedPhotos'
output_json = '/content/drive/MyDrive/analys_masks.json'
token_log_path = '/content/drive/MyDrive/token_usage_summary.txt'

MAX_PIXELS = 313600

extraction_prompt = """
Extract all text from this turbocharger nameplate.
Maintain the line structure.
If you see serial numbers or part numbers, ensure they are exact.
CRITICAL: The text on this nameplate consists ONLY of Latin letters, numbers, hyphens (-), and slashes (/). You are strictly forbidden from outputting any other symbols like cyrillic letters or punctuation.
Return ONLY the extracted text, no introductory sentences.
"""

print("Loading Qwen2-VL model...")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-7B-Instruct",
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
    attn_implementation="sdpa"
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-7B-Instruct")

print("Warming up the GPU...")
dummy_input = processor(text=["Warm up"], return_tensors="pt").to("cuda")
with torch.no_grad():
    model.generate(**dummy_input, max_new_tokens=5)
torch.cuda.synchronize()
print("GPU Warm-up complete.")

if os.path.exists(output_json):
    os.remove(output_json)
    print("🗑️ Deleted existing JSON file. Starting fresh!")

results = {}
valid_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp')

total_session_input_tokens = 0
total_session_output_tokens = 0
total_gpu_time = 0.0

if not os.path.exists(folder_path):
    print(f"Error: The folder '{folder_path}' does not exist.")
else:
    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(valid_extensions)]
    print(f"Found {len(image_files)} images total. Starting extraction...")

    for i, filename in enumerate(image_files, start=1):
        img_path = os.path.join(folder_path, filename)
        print(f"\n[{i}/{len(image_files)}] Processing: {filename}")

        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": img_path,
                        "max_pixels": MAX_PIXELS
                    },
                    {
                        "type": "text",
                        "text": extraction_prompt
                    },
                ],
            }
        ]

        start_event = torch.cuda.Event(enable_timing=True)
        end_event = torch.cuda.Event(enable_timing=True)
        start_event.record()

        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)

        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        inputs = inputs.to("cuda")

        input_token_count = inputs.input_ids.shape[1]
        total_session_input_tokens += input_token_count

        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=512)

        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]

        output_token_count = len(generated_ids_trimmed[0])
        total_session_output_tokens += output_token_count

        total_tokens = input_token_count + output_token_count
        print(f"📊 Tokens -> Input: {input_token_count} | Output: {output_token_count} | Total: {total_tokens}")

        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]

        end_event.record()
        torch.cuda.synchronize()

        image_time = start_event.elapsed_time(end_event) / 1000.0
        total_gpu_time += image_time
        print(f"⏱️ True GPU Extraction Time for {filename}: {image_time:.2f} seconds")

        results[filename] = output_text.strip()

        os.makedirs(os.path.dirname(output_json), exist_ok=True)
        with open(output_json, 'w', encoding='utf-8') as json_file:
            json.dump(results, json_file, ensure_ascii=False, indent=4)

        torch.cuda.empty_cache()

    print(f"\n✅ Extraction complete! All data saved to: {output_json}")

    grand_total_tokens = total_session_input_tokens + total_session_output_tokens

    summary_text = (
        f"--- Execution & Token Usage Summary ---\n"
        f"Total Images Processed: {len(image_files)}\n"
        f"Total Input Tokens:     {total_session_input_tokens}\n"
        f"Total Output Tokens:    {total_session_output_tokens}\n"
        f"Grand Total Tokens:     {grand_total_tokens}\n"
        f"Total GPU Compute Time: {total_gpu_time:.2f} seconds\n"
        f"Average Time per Image: {(total_gpu_time / len(image_files)):.2f} seconds\n"
        f"---------------------------------------\n"
    )

    print(f"\n{summary_text}")

    os.makedirs(os.path.dirname(token_log_path), exist_ok=True)
    with open(token_log_path, 'w', encoding='utf-8') as f:
        f.write(summary_text)

    print(f"📄 Summary successfully exported to: {token_log_path}")

In [ ]:
end_event.record()

torch.cuda.synchronize()
gpu_time_seconds = start_event.elapsed_time(end_event) / 1000.0

print(f"Total GPU Computational Time: {gpu_time_seconds:.2f} seconds")

Constrained Prompt

In [ ]:
import os
import json
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info


folder_path = "/content/drive/MyDrive/allRotatedPhotos"
output_json = '/content/drive/MyDrive/analys_masks.json'
token_log_path = '/content/drive/MyDrive/token_usage_summary.txt'


MAX_PIXELS = 313600
SAVE_INTERVAL = 10

extraction_prompt = """
Analyze this a Garrett turbocharger nameplate and extract the information into a strict JSON format.

Follow these two rules:
1. "raw_text": Extract all visible text from the nameplate. Keep it exact, with no introductory text, descriptions, or added formatting. The text on this nameplate consists ONLY of Latin letters, numbers, hyphens (-), and slashes (/). You are strictly forbidden from outputting any other symbols like cyrillic letters or punctuation.
2. "part_number": Extract ONLY the Garrett Part Number. It must follow the format of 6 digits, a hyphen (-), and 1 to 4 digits (e.g., 758219-3). It MUST start with the digit 4, 7, or 8. If you cannot find a matching sequence, return null.

CRITICAL: Return ONLY valid JSON. Do not include markdown backticks (```) or any other conversational text.

Expected format:
{
  "raw_text": "all the text from the plate goes here...",
  "part_number": "XXXXXX-X..."
}
"""

print("Loading Qwen2-VL model...")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-7B-Instruct",
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
    attn_implementation="sdpa"
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-7B-Instruct")


if os.path.exists(output_json):
    os.remove(output_json)
    print("🗑️ Deleted existing JSON file. Starting fresh!")

results = []
valid_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp')

total_session_input_tokens = 0
total_session_output_tokens = 0

if not os.path.exists(folder_path):
    print(f"Error: The folder '{folder_path}' does not exist.")
else:
    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(valid_extensions)]
    print(f"Found {len(image_files)} images total. Starting extraction...")

    for i, filename in enumerate(image_files, start=1):
        img_path = os.path.join(folder_path, filename)
        print(f"\n[{i}/{len(image_files)}] Processing: {filename}")

        try:
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": img_path, "max_pixels": MAX_PIXELS},
                        {"type": "text", "text": extraction_prompt},
                    ],
                }
            ]

            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            image_inputs, video_inputs = process_vision_info(messages)

            inputs = processor(
                text=[text],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt",
            ).to("cuda")

            input_token_count = inputs.input_ids.shape[1]
            total_session_input_tokens += input_token_count # Add to global tracker

            with torch.no_grad():
                generated_ids = model.generate(**inputs, max_new_tokens=512)

            generated_ids_trimmed = [
                out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
            ]

            output_token_count = len(generated_ids_trimmed[0])
            total_session_output_tokens += output_token_count
            total_tokens = input_token_count + output_token_count
            print(f"📊 Tokens -> Input: {input_token_count} | Output: {output_token_count} | Total: {total_tokens}")

            output_text = processor.batch_decode(
                generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
            )[0].strip()

            if output_text.startswith("```json"):
                output_text = output_text[7:]
            if output_text.startswith("```"):
                output_text = output_text[3:]
            if output_text.endswith("```"):
                output_text = output_text[:-3]

            output_text = output_text.strip()

            try:
                parsed_data = json.loads(output_text)

                results.append({
                    "image_name": filename,
                    "part_number": parsed_data.get("part_number"),
                    "raw_text": parsed_data.get("raw_text")
                })

            except json.JSONDecodeError:
                print(f"⚠️ Failed to parse JSON for {filename}. Raw output: {output_text}")
                results.append({
                    "image_name": filename,
                    "part_number": "ERROR",
                    "raw_text": output_text
                })

        except Exception as e:
            print(f"⚠️ Critical Error processing {filename}: {e}")
            results.append({
                "image_name": filename,
                "part_number": "CRITICAL_ERROR",
                "raw_text": str(e)
            })

        finally:
            torch.cuda.empty_cache()

        if i % SAVE_INTERVAL == 0 or i == len(image_files):
            os.makedirs(os.path.dirname(output_json), exist_ok=True)
            with open(output_json, 'w', encoding='utf-8') as json_file:
                json.dump(results, json_file, ensure_ascii=False, indent=4)
            print(f"💾 Checkpoint saved at image {i}")

    print(f"\n✅ Extraction complete! All data saved to: {output_json}")

    grand_total_tokens = total_session_input_tokens + total_session_output_tokens

    summary_text = (
        f"--- Token Usage Summary ---\n"
        f"Total Images Processed: {len(image_files)}\n"
        f"Total Input Tokens:     {total_session_input_tokens}\n"
        f"Total Output Tokens:    {total_session_output_tokens}\n"
        f"Grand Total Tokens:     {grand_total_tokens}\n"
        f"---------------------------\n"
    )

    print(f"\n{summary_text}")

    os.makedirs(os.path.dirname(token_log_path), exist_ok=True)
    with open(token_log_path, 'w', encoding='utf-8') as f:
        f.write(summary_text)

    print(f"📄 Token usage summary successfully exported to: {token_log_path}")